In [ ]:
# Installs required packages
!pip install fastai>=2.7 scikit-image tqdm -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

While scrolling through the COCO dataset I noticed grayscale images, so I tried to remove them with this cell of code. The problem is that some images were almost grayscale, and the code as it is would not work for them so the only (unfortunate) soultion was to manually delete the unrealistic images (more details in the next cell)

In [ ]:
# Grayscale photos removal from dataset
DRIVE_DATASET_PATH = "/content/drive/MyDrive/datasets/coco" # we used the COCO dataset

import os
import glob
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

def is_grayscale_image(img_path):
    """Check if image is grayscale by comparing RGB channels"""
    try:
        img = Image.open(img_path).convert("RGB") # convert image to RGB mode
        img_array = np.array(img)

        # Check if all channels are identical
        r, g, b = img_array[:,:,0], img_array[:,:,1], img_array[:,:,2] # separate R, G, B channels
        return np.array_equal(r, g) and np.array_equal(g, b) # an image is grayscale if all channels are identical
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        raise e

# Get all image paths (with common extensions)
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
all_image_paths = []

for ext in image_extensions:
    # Match files in the immediate directory
    all_image_paths.extend(glob.glob(os.path.join(DRIVE_DATASET_PATH, ext)))
    # Match files in all subdirectories (recursive depth search)
    all_image_paths.extend(glob.glob(os.path.join(DRIVE_DATASET_PATH, '**', ext), recursive=True))

# Remove duplicates
all_image_paths = list(set(all_image_paths))

if len(all_image_paths) == 0:
    print(f"No images found in {DRIVE_DATASET_PATH}. Please check the path!")
else:
    print(f"Found {len(all_image_paths)} images in dataset!")
    print("Deleting grayscale images...\n")

    grayscale_count = 0
    color_count = 0
    error_count = 0

    # Process each image
    for img_path in tqdm(all_image_paths, desc="Processing images"): # tqdm - shows progress meter
        try:
            if is_grayscale_image(img_path):
                # Delete grayscale image from Drive for dataset correction
                os.remove(img_path)
                grayscale_count += 1
            else:
                color_count += 1
        except Exception as e:
            error_count += 1
            print(f"\nError processing {os.path.basename(img_path)}: {e}")

    print("-" * 40)
    print(f"Processing complete! Cleanup summary:")
    print(f"   Processed:      {len(all_image_paths)}")
    print(f"   Deleted (gray): {grayscale_count}")
    print(f"   Kept (color):   {color_count}")
    print(f"   Errors:         {error_count}")
    print("-" * 40)

This cell creates a new folder in Google Drive with 16000 valid images from the original dataset. I only used this dataset in my training because I noticed some edited or filtered images in the COCO dataset that resulted in unrealistic coloring, which would impact our model. So I manually processed the created subfolder and eliminated as many of these images as I could. The 16k image extracion was so I could guarantee I will have more than 13k images remaining after the manual deletion process.

In [ ]:
# Create a subset dataset folder
import os
import glob
import shutil
import numpy as np
from tqdm.notebook import tqdm

ORIGINAL_DATASET_PATH = "/content/drive/MyDrive/datasets/coco"
SUBSET_DATASET_PATH = "/content/drive/MyDrive/datasets/coco_subset_16000"

NUM_IMAGES_TO_EXTRACT = 16000

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
all_image_paths = []

# Collect all image paths recursively
for ext in image_extensions:
    # Using recursive search to ensure all subfolders are covered
    all_image_paths.extend(glob.glob(os.path.join(ORIGINAL_DATASET_PATH, '**', ext), recursive=True))

# Remove duplicates
all_image_paths = list(set(all_image_paths))

if len(all_image_paths) == 0:
    print(f"No images found in {ORIGINAL_DATASET_PATH}. Please check the path!")
else:
    print(f"Found {len(all_image_paths)} images in dataset!")

    # Shuffle images with random-sampling with seed 42
    np.random.seed(42)
    np.random.shuffle(all_image_paths)

    # Select the first N images after shuffling
    selected_paths = all_image_paths[:NUM_IMAGES_TO_EXTRACT]

    # Create the destination directory if it doesn't exist
    os.makedirs(SUBSET_DATASET_PATH, exist_ok=True)

    print(f"Copying {len(selected_paths)} images to: {SUBSET_DATASET_PATH}...")

    copy_count = 0
    error_count = 0

    for img_path in tqdm(selected_paths, desc="Copying files"):
        try:
            filename = os.path.basename(img_path) # extract file name
            dest_path = os.path.join(SUBSET_DATASET_PATH, filename) # create file with the above file name in the destination folder
            # copy the image to the destination path
            shutil.copy2(img_path, dest_path)  # shutil.copy2 preserves original file metadata
            copy_count += 1
        except Exception as e:
            error_count += 1
            print(f"\nError copying {filename}: {e}")

    print("-" * 40)
    print("Execution Summary:")
    print(f"  Images to copy: {len(all_image_paths)}")
    print(f"  Copied:         {copy_count}")
    print(f"  Errors:         {error_count}")
    print("-" * 40)

Count the images left in dataset subfolder after manually filtering them

In [ ]:
# Count images in dataset folder
import os
import glob

SUBSET_FOLDER_PATH = "/content/drive/MyDrive/datasets/coco_subset_16000"

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
all_image_paths = []

for ext in image_extensions:
    all_image_paths.extend(glob.glob(os.path.join(SUBSET_FOLDER_PATH, ext)))
    all_image_paths.extend(glob.glob(os.path.join(SUBSET_FOLDER_PATH, '**', ext), recursive=True))

all_image_paths = list(set(all_image_paths))

if len(all_image_paths) == 0:
    print(f"No images found in {SUBSET_FOLDER_PATH}. Please check the path!")
else:
    print(f"Found {len(all_image_paths)} images in dataset!")

    # Count by extension
    extensions_count = {}
    for img_path in all_image_paths:
        ext = os.path.splitext(img_path)[1].lower() # split text into root and extension, and get the ext
        extensions_count[ext] = extensions_count.get(ext, 0) + 1 # dictionary of extension frequency

    print(f"\nDataset distribution:")
    for ext, count in sorted(extensions_count.items()):
        print(f"   {ext}: {count} images")